# Free Text-to-Video with ComfyUI + Wan 2.1 (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harekrishnashah13/website_deployemnt/blob/claude/video-generation-text-039bcd/notebooks/ComfyUI_Wan2.1_Text_to_Video_Colab.ipynb)

This notebook runs the **"Wan 2.1 Text to Video"** workflow from the open-source
[`Comfy-Org/workflow_templates`](https://github.com/Comfy-Org/workflow_templates)
repository on top of [ComfyUI](https://github.com/comfyanonymous/ComfyUI) — both open source, both free.

It is the lightest fully-local (non-API) text-to-video workflow in that repo (~9.8GB of
model weights total), so it fits comfortably on Colab's **free T4 GPU** (15GB VRAM).

**Before running:** `Runtime` -> `Change runtime type` -> Hardware accelerator: **T4 GPU** (free tier).

Then `Runtime` -> `Run all`. First run takes a while (downloading ~9.8GB of models);
generation itself typically takes a few minutes per video on a T4.


## 1. Check GPU

In [ ]:
!nvidia-smi


## 2. Install ComfyUI

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt


## 3. Download the Wan 2.1 Text-to-Video models

Model files and URLs are taken directly from the
[`text_to_video_wan.json`](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/text_to_video_wan.json)
workflow template.

In [ ]:
import os

os.makedirs('models/diffusion_models', exist_ok=True)
os.makedirs('models/text_encoders', exist_ok=True)
os.makedirs('models/vae', exist_ok=True)

MODELS = {
    'models/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors',
    'models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
    'models/vae/wan_2.1_vae.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors',
}

for dest, url in MODELS.items():
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        print('Already downloaded:', dest)
        continue
    print('Downloading', dest, '...')
    !wget -q --show-progress -O "{dest}" "{url}"


## 4. Set your prompt

In [ ]:
positive_prompt = "a fox moving quickly in a beautiful winter scenery, nature, trees, mountains, daytime, tracking camera"  #@param {type:"string"}
negative_prompt = "blurry, low quality, static, worst quality, low quality, deformed, disfigured, extra fingers, poorly drawn hands, poorly drawn face, cluttered background"  #@param {type:"string"}
width = 832  #@param {type:"integer"}
height = 480  #@param {type:"integer"}
num_frames = 33  #@param {type:"integer"}
steps = 30  #@param {type:"integer"}
cfg = 6  #@param {type:"number"}
seed = -1  #@param {type:"integer"}


## 5. Start the ComfyUI server

In [ ]:
import subprocess, time, requests

server_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'],
    cwd='/content/ComfyUI', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for _ in range(120):
    try:
        r = requests.get('http://127.0.0.1:8188/system_stats', timeout=2)
        if r.status_code == 200:
            print('ComfyUI server is up.')
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(2)
else:
    out = server_proc.stdout.read() if server_proc.stdout else ''
    raise RuntimeError('ComfyUI server did not start in time.\n' + out)


## 6. Build and queue the workflow

This is the `text_to_video_wan` workflow graph, translated into ComfyUI's API prompt format,
with your parameters from step 4 substituted in.

In [ ]:
import random, uuid, time, requests

run_seed = seed if seed is not None and seed >= 0 else random.randint(0, 2**32 - 1)

workflow = {
    "37": {"class_type": "UNETLoader", "inputs": {
        "unet_name": "wan2.1_t2v_1.3B_fp16.safetensors", "weight_dtype": "default"}},
    "38": {"class_type": "CLIPLoader", "inputs": {
        "clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan", "device": "default"}},
    "39": {"class_type": "VAELoader", "inputs": {"vae_name": "wan_2.1_vae.safetensors"}},
    "40": {"class_type": "EmptyHunyuanLatentVideo", "inputs": {
        "width": width, "height": height, "length": num_frames, "batch_size": 1}},
    "6": {"class_type": "CLIPTextEncode", "inputs": {"text": positive_prompt, "clip": ["38", 0]}},
    "7": {"class_type": "CLIPTextEncode", "inputs": {"text": negative_prompt, "clip": ["38", 0]}},
    "48": {"class_type": "ModelSamplingSD3", "inputs": {"shift": 8, "model": ["37", 0]}},
    "3": {"class_type": "KSampler", "inputs": {
        "seed": run_seed, "steps": steps, "cfg": cfg, "sampler_name": "uni_pc",
        "scheduler": "simple", "denoise": 1,
        "model": ["48", 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["40", 0]}},
    "8": {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["39", 0]}},
    "49": {"class_type": "CreateVideo", "inputs": {"images": ["8", 0], "fps": 16}},
    "50": {"class_type": "SaveVideo", "inputs": {
        "video": ["49", 0], "filename_prefix": "video/ComfyUI", "format": "auto", "codec": "auto"}},
}

client_id = str(uuid.uuid4())
resp = requests.post('http://127.0.0.1:8188/prompt', json={'prompt': workflow, 'client_id': client_id})
resp.raise_for_status()
prompt_id = resp.json()['prompt_id']
print('Queued prompt:', prompt_id, '- generation can take several minutes on a free T4 GPU.')

output_files = []
max_wait_s = 1800
waited = 0
while not output_files and waited < max_wait_s:
    time.sleep(5)
    waited += 5
    hist = requests.get(f'http://127.0.0.1:8188/history/{prompt_id}').json()
    entry = hist.get(prompt_id)
    if not entry:
        continue
    status = entry.get('status', {})
    if status.get('status_str') == 'error':
        raise RuntimeError(f'Generation failed: {status}')
    if status.get('completed'):
        for node_out in entry.get('outputs', {}).values():
            for val in node_out.values():
                if isinstance(val, list):
                    for item in val:
                        if isinstance(item, dict) and 'filename' in item:
                            output_files.append(item)
        break

if not output_files:
    raise RuntimeError('Timed out waiting for video generation.')
print('Done:', output_files)


## 7. Preview and download your video

In [ ]:
import os
from IPython.display import Video, display

vid = output_files[0]
subfolder = vid.get('subfolder', '')
video_path = os.path.join('/content/ComfyUI/output', subfolder, vid['filename'])
print(video_path)
display(Video(video_path, embed=True))

try:
    from google.colab import files
    files.download(video_path)
except ImportError:
    print('Not running in Colab - find your video at', video_path)
